In [1]:
import pandas as pd 
import numpy as np
import CoolProp.CoolProp as CP
from CoolProp.CoolProp import PropsSI
import math
import torch
df_database = pd.read_excel('./features.xlsx', engine='openpyxl', engine_kwargs={'data_only': False})
class CoolProp_Sample():
    def __init__(self, molecule):
        # 寻找对应的行
        self.row = df_database[df_database['Compound name ↓'] == molecule]
        self.cas_ = str(self.row['CAS no'].values[0])[2:-1]
        self.t_crit = PropsSI('TCRIT', self.cas_)
        self.p_crit = PropsSI('PCRIT', self.cas_)
        self.AS = CP.AbstractState("HEOS", self.cas_)

    @property
    def CAS(self):
        return self.cas_

    def Tr(self, T):
        return T / self.t_crit
    
    def Pr(self, P):
        return P / self.p_crit

    def Z(self, T, P, phase = None):
        self.AS.unspecify_phase()
        try:
            if phase is not None:
                if T < self.t_crit and P < self.p_crit:
                    self.AS.specify_phase(phase)
                if T > self.t_crit and phase == CP.iphase_liquid:
                    return np.nan
                if P > self.p_crit and phase == CP.iphase_gas:
                    return np.nan
            self.AS.update(CP.PT_INPUTS, P, T)
            Z = self.AS.compressibility_factor()
            Z_ = torch.tensor(Z, dtype=torch.float32)
            if torch.isnan(Z_) or torch.isinf(Z_) or Z_ <= 6e-12:
                # print(f"Invalid Z value for {self.cas_} at T={T} K and P={P} Pa: Z={Z}")
                return np.nan
            return Z
        except Exception as e:
            # print(f"Error calculating Z for {self.cas_} at T={T} K and P={P} Pa: {str(e)}")
            return np.nan
    
    @property
    def T_min(self):
        return PropsSI('TMIN', self.cas_)
    
    @property
    def T_max(self):
        return PropsSI('TMAX', self.cas_)
    
    @property
    def P_max(self):
        return PropsSI('PMAX', self.cas_)
    
    @property
    def T_crit(self):
        return self.t_crit

    @property
    def P_crit(self):
        return self.p_crit

    @property
    def P_min(self):
        return PropsSI('PMIN', self.cas_)

    @property
    def T_triple(self):
        return PropsSI('TTRIPLE', self.cas_)

    def SaturatedVaporPressure(self, T):
        self.AS.unspecify_phase()
        self.AS.update(CP.QT_INPUTS, 1, T)
        return self.AS.p()

    def fugacity_coefficient(self, T, P, phase = None):
        self.AS.unspecify_phase()
        return_phase = 0
        try:
            if phase is not None:
                if T < self.t_crit and P < self.p_crit:
                    self.AS.specify_phase(phase)
                if T > self.t_crit and phase == CP.iphase_liquid:
                    return np.nan, np.nan
                if P > self.p_crit and phase == CP.iphase_gas:
                    return np.nan, np.nan
            self.AS.update(CP.PT_INPUTS, P, T)
        except Exception as e:
            # print(f"Error updating state for {self.cas_} at T={T} K and P={P} Pa: {str(e)}")
            return 0, np.nan
        if phase is not None:
            return_phase = phase == CP.iphase_liquid
        else:
            if T < self.t_crit and P >= self.p_crit:
                return_phase = 1
            else:
                return_phase = self.AS.phase() == CP.iphase_liquid
        phi = self.AS.fugacity_coefficient(0)
        phi_ = torch.tensor(phi, dtype=torch.float32)
        if torch.isnan(phi_) or torch.isinf(phi_) or phi_ <= 6e-12:
            # print(f"Invalid fugacity coefficient for {self.cas_} at T={T} K and P={P} Pa: phi={phi}")
            return 0, np.nan
        return return_phase, math.log(phi)

    def fugacity_coefficient_Saturation(self, T, Q = 1):
        self.AS.unspecify_phase()
        try:
            self.AS.update(CP.QT_INPUTS, Q, T)
        except Exception as e:
            # print(f"Error updating state for {self.cas_} at T={T} K and Q={Q}: {str(e)}")
            return np.nan
        phi = self.AS.fugacity_coefficient(0)
        phi_ = torch.tensor(phi, dtype=torch.float32)
        if torch.isnan(phi_) or torch.isinf(phi_) or phi_ <= 6e-12:
            # print(f"Invalid fugacity coefficient for {self.cas_} at T={T} K and Q={Q}: phi={phi}")
            return np.nan
        return math.log(phi)
    
    def phase(self, T, P):
        AS = CP.AbstractState("HEOS", self.cas_)
        try:
            AS.update(CP.PT_INPUTS, P, T)
        except Exception as e:
            # print(f"Error updating state for {self.cas_} at T={T} K and P={P} Pa: {str(e)}")
            return "unknown"
        return AS.phase()


In [2]:
import pandas as pd
import torch
import CoolProp.CoolProp as CP
# 打开features.xlsx
features = pd.read_excel('features.xlsx')
# 计算特征的mean和std
feature_stats = pd.read_excel('feature_statistics.xlsx', index_col=0)
mean_ = feature_stats['mean'].values
std_ = feature_stats['std'].values
Names = features['Compound name ↓'].tolist()
print(len(features))
size_ = 4000 + 2500 + 100*20
database = torch.zeros((len(features) * size_, 31 + 2 + 4 + 1))
j = 0
count = 0
for name in Names:
    w = CoolProp_Sample(name)
    T_min = w.T_min #max(w.T_min + 0.5, w.TPT)
    T_max = w.T_max
    P_max = w.P_max
    P_min = w.P_min #max(w.P_min + 0.5, w.TPP)
    T_crit = w.T_crit
    P_crit = w.P_crit
    ratiot = (T_max - min(T_crit, T_max)) / 50
    # 在features中找出name对应的行
    row = features[features['Compound name ↓'] == name]
    fea = row.iloc[0, 2:].values
    fea = (fea - mean_) / std_
    fea = np.array(fea, dtype=np.float32)
    database[j*size_:(j+1)*size_, 0:31] = torch.from_numpy(fea).reshape(1, -1)
    database[j*size_:(j+1)*size_, 31:33] = torch.tensor([T_crit, P_crit])
    pt_right = []
    pt_left = []
    k = 0
    T_inf = T_min
    T_max_uesd = T_crit
    T = np.linspace(T_inf, T_max_uesd, 40, dtype=np.double)
    P = []
    for t in T:
        P.append(w.SaturatedVaporPressure(t))
    ratiop = P_max / P_crit
    if np.abs(ratiop) < 1e-6:
        ratiop = P_crit / P[0]
    print(f"Processing {name}... ratiot={ratiot}, ratiop={ratiop}")
    for i in range(1,51):
        for t, p in zip(T, P):
            database[j*size_ + k, 33:] = torch.tensor([t + ratiot * i, p, w.Z(t + ratiot * i, p), *w.fugacity_coefficient(t + ratiot * i, p)])
            k += 1
        pt_left.append(T[-1] + ratiot * i)
    for t, p in zip(T, P):
        newp = np.exp(np.linspace(np.log(p), np.log(p * ratiop), 51)[1:])
        for p_ in newp:
            database[j*size_ + k, 33:] = torch.tensor([t, p_, w.Z(t, p_), *w.fugacity_coefficient(t, p_)])
            k += 1
    pt_right = np.exp(np.linspace(np.log(P[-1]), np.log(P[-1] * ratiop), 50))
    # pt_left和pt_right二维网格列表，pt_left只取T，pt_right只取P
    for t_left in pt_left:
        for p_right in pt_right:
            database[j*size_ + k, 33:] = torch.tensor([t_left, p_right, w.Z(t_left, p_right), *w.fugacity_coefficient(t_left, p_right)])
            k += 1
    T = np.linspace(T_inf, T_crit - 0.1, 100, dtype=np.double)
    # 依据T,p产生一个围绕其10%误差的条带，P每隔0.5%取一个点，形成一个条带
    for t in T:
        p = w.SaturatedVaporPressure(t)
        for i in range(-10, 11):
            if i == 0:
                continue
            p_ = p * (1 + i * 0.01)
            Z = w.Z(t, p_)
            if i < 0:
                Z_ref = w.Z(t, p_, CP.iphase_gas)
            else:
                Z_ref = w.Z(t, p_, CP.iphase_liquid)
            if not np.isnan(Z) and not np.isnan(Z_ref):
                assert np.abs(Z_ref - Z) < 1e-6, f"Error: Both Z and Z_ref are not equal for {name} at T={t} K and P={p_} Pa"
            database[j*size_ + k, 33:] = torch.tensor([t, p_, w.Z(t, p_), *w.fugacity_coefficient(t, p_)])
            k += 1
    j += 1
    # 检查k/j是否等于size_
    assert k == size_, f"Error: k={k} does not equal size_={size_} for {name}"
    # break
# 删除存在nan的行
database = database[~torch.isnan(database).any(dim=1)]
torch.save(database, 'database_new_max.pt')

101
Processing METHANE... ratiot=8.688719946974262, ratiop=217.4290956855972
Processing ETHANE... ratiot=7.39355999999969, ratiop=184.7214818981554
Processing PROPANE... ratiot=5.602199820980732, ratiop=235.22961890249312
Processing ISOBUTANE... ratiot=3.343799999999908, ratiop=9.644530129353019
Processing N-BUTANE... ratiot=2.9974999999998397, ratiop=3.1612223248010416
Processing N-PENTANE... ratiot=3.6060000025776673, ratiop=231.62452945240986
Processing 2-METHYL-BUTANE... ratiot=0.793004105771082, ratiop=296.01412035981565
Processing 2,2-DIMETHYL-PROPANE... ratiot=2.3252081633731425, ratiop=62.57239901736332
Processing N-HEXANE... ratiot=1.8436000023252985, ratiop=30.22224529544334
Processing 2-METHYL-PENTANE... ratiot=1.045982044977685, ratiop=328.6599320798655
Processing N-HEPTANE... ratiot=1.1754816982137481, ratiop=36.05131035530953
Processing N-OCTANE... ratiot=3.2252000010828485, ratiop=402.64275413046323
Processing N-NONANE... ratiot=0.10904373576165653, ratiop=350.5833439507

In [3]:
import pandas as pd
import torch
# 打开features.xlsx
features = pd.read_excel('features.xlsx')
feature_stats = pd.read_excel('feature_statistics.xlsx', index_col=0)
mean_ = feature_stats['mean'].values
std_ = feature_stats['std'].values
Names = features['Compound name ↓'].tolist()
print(len(features))
points = 40
size_ = 2 * points * points
database = torch.zeros((len(features) * size_, 31 + 2 + 4 + 1), dtype=torch.float32)
j = 0
for name in Names:
    w = CoolProp_Sample(name)
    T_min = w.T_min
    T_max = w.T_max
    P_max = w.P_max
    P_min = w.P_min
    T_crit = w.T_crit
    P_crit = w.P_crit
    ratiot = (T_max - min(T_crit, T_max)) / 50
    print(f"Processing {name}... ratiot={ratiot}, ratiop={ratiop}")
    # 在features中找出name对应的行
    row = features[features['Compound name ↓'] == name]
    fea = row.iloc[0, 2:].values
    fea = (fea - mean_) / std_
    fea = np.array(fea, dtype=np.double)
    database[j*size_:(j+1)*size_, 0:31] = torch.from_numpy(fea).reshape(1, -1)
    database[j*size_:(j+1)*size_, 31:33] = torch.tensor([w.T_crit, w.P_crit])
    pt_right = []
    pt_left = []
    k = 0
    T_inf = T_min
    T_max_uesd = T_crit
    T = np.linspace(T_inf, T_max_uesd, 80, dtype=np.double)
    P = []
    for t in T:
        P.append(w.SaturatedVaporPressure(t))
    ratiop = P_max / P_crit
    if np.abs(ratiop) < 1e-6:
        ratiop = P_crit / P[0]
    mid = len(T) // 2
    # 在右下角采样，与(T_max, P_min)围成的矩形区域内采样
    midT = T[mid] + ratiot * 50
    midP = P[mid]
    
    T0 = np.linspace(midT, T_max, points, dtype=np.double)
    P0 = np.exp(np.linspace(np.log(P_min), np.log(midP), points, dtype=np.double))
    for t in T0:
        for p in P0:
            database[j*size_ + k, 33:] = torch.tensor([t, p, w.Z(t, p), *w.fugacity_coefficient(t, p)])
            k += 1

    midT = T[mid]
    midP = P[mid] * ratiop
    # 在左上角采样，与(T_min, P_max)围成的矩形区域内采样
    T0 = np.linspace(T_inf, midT, points, dtype=np.double)
    P0 = np.exp(np.linspace(np.log(midP), np.log(P_max), points, dtype=np.double))
    for t in T0:
        for p in P0:
            database[j*size_ + k, 33:] = torch.tensor([t, p, w.Z(t, p), *w.fugacity_coefficient(t, p)])
            k += 1
    j += 1
    # break
# 删除存在nan的行
database = database[~torch.isnan(database).any(dim=1)]
torch.save(database, 'database_new_max_val.pt')

101
Processing METHANE... ratiot=8.688719946974262, ratiop=113.6354693355578
Processing ETHANE... ratiot=7.39355999999969, ratiop=217.4290956855972
Processing PROPANE... ratiot=5.602199820980732, ratiop=184.7214818981554
Processing ISOBUTANE... ratiot=3.343799999999908, ratiop=235.22961890249312
Processing N-BUTANE... ratiot=2.9974999999998397, ratiop=9.644530129353019
Processing N-PENTANE... ratiot=3.6060000025776673, ratiop=3.1612223248010416
Processing 2-METHYL-BUTANE... ratiot=0.793004105771082, ratiop=231.62452945240986
Processing 2,2-DIMETHYL-PROPANE... ratiot=2.3252081633731425, ratiop=296.01412035981565
Processing N-HEXANE... ratiot=1.8436000023252985, ratiop=62.57239901736332
Processing 2-METHYL-PENTANE... ratiot=1.045982044977685, ratiop=30.22224529544334
Processing N-HEPTANE... ratiot=1.1754816982137481, ratiop=328.6599320798655
Processing N-OCTANE... ratiot=3.2252000010828485, ratiop=36.05131035530953
Processing N-NONANE... ratiot=0.10904373576165653, ratiop=402.64275413046